In [ ]:
import json
import math
import re
from pathlib import Path

import pandas as pd
import yaml

WORKSPACE = Path("/workspace")
CONFIGS_DIR = WORKSPACE / "configs"
CHECKPOINTS_DIR = WORKSPACE / "checkpoints"

# Order matches the paper appendix (sorted by nx, ties broken by appendix order).
# Each entry: (system_key, display_name).
SYSTEMS = [
    ("inverted_pendulum",    "Inverted Pendulum"),
    ("double_integrator_1d", "Double Integrator (1D)"),
    ("vertical_drone_2d",    "Vertical Drone (2D)"),
    ("dubins_car",           "Dubins Car"),
    ("double_integrator_2d", "Double Integrator (2D)"),
    ("kinematic_bicycle",   "Kinematic Bicycle"),
    ("cart_pole",            "Cart-Pole"),
    ("dynamic_unicycle",     "Dynamic Unicycle"),
    ("relative_unicycle",    "Relative Unicycle"),
    ("double_integrator_3d", "Double Integrator (3D)"),
    ("manipulator_3dof",     "3-DOF Manipulator"),
    ("landing_rocket",       "Landing Rocket"),
    ("quadruped_trunk",      "Quadruped Trunk"),
    ("auv_6dof",             "6-DoF Underwater Vehicle"),
    ("quadrotor",            "Quadrotor"),
]

# The three methods compared in the paper.  Acronyms below are used both in
# the printed DataFrame and in the LaTeX sub-header.
#   ND   — no data: PDE/HJB residual loss only (no supervision targets).
#   FCD  — full-control data (mppi).
#   VRCD — vertex-restricted control data (beam / stochastic beam / B&B).
METHODS = [
    ("ND",          "NO_DATA"),
    ("FCD",         "FC_DATA"),
    ("VRCD (ours)", "VRC_DATA"),
]

# Per-metric columns rendered in the table.  Each entry:
#   (json_key, pandas_label, latex_label, fmt_kind, best_direction)
# Arrows in the labels indicate whether smaller (↓) or larger (↑) is better.
#   rho_FS — false-safe rate (%, smaller = safer certificates).
#   rho_FU — false-unsafe rate (%, smaller = less conservative).
#   eta_eff — effective safe volume = PV * (1 - FS) as a fraction of the
#             state-space bounding box (larger = larger trustworthy safe set).
METRICS = [
    ("fs",    "FS (%) ↓", "$\\rho_\\mathrm{FS}$ (\\%) $\\downarrow$", "pct", "min"),
    ("fu",    "FU (%) ↓", "$\\rho_\\mathrm{FU}$ (\\%) $\\downarrow$", "pct", "min"),
    ("v_eff", "η_eff ↑",  "$\\eta_\\mathrm{eff}$ $\\uparrow$",        "vol", "max"),
]

MISSING = "--"  # cell placeholder when an artifact is missing


def _system_dims(system_key):
    """Return (nx, nu) read from the config YAML."""
    cfg_path = CONFIGS_DIR / f"{system_key}.yaml"
    if not cfg_path.is_file():
        return None, None
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    sys_cfg = cfg.get("system", {})
    nx = len(sys_cfg.get("x_min", []))
    nu = len(sys_cfg.get("u_min", []))
    return nx, nu


def _is_nan(x):
    return isinstance(x, float) and math.isnan(x)


def _load_validation(system_key, group_dir):
    """Load a single validation.json artifact and pull out the metrics.

    Returns a dict with keys ``fs``, ``fu``, ``v_eff``; any missing field
    is reported as ``None``.

    ``v_eff = PV * (1 - FS/100)`` is the volume of the predicted safe set
    intersected with the truly-safe set, as a fraction of the state-box.
    When PV is exactly 0 (predicted-safe set is empty under the current
    model), ``v_eff`` is 0 regardless of whether FS could be evaluated.
    """
    out = {"fs": None, "fu": None, "v_eff": None}
    val_path = CHECKPOINTS_DIR / group_dir / system_key / "validation.json"
    if not val_path.is_file():
        return out
    try:
        with open(val_path) as f:
            data = json.load(f)
    except (OSError, json.JSONDecodeError):
        return out
    sm = data.get("stratified_metrics", {})
    out["fs"] = sm.get("predicted_safe", {}).get("false_safe_rate")
    out["fu"] = sm.get("predicted_unsafe", {}).get("false_unsafe_rate")
    vm = data.get("volume_metrics", {})
    pv = vm.get("predicted_safe_vol_ratio")
    fs = out["fs"]
    if pv is None or _is_nan(pv):
        out["v_eff"] = None
    elif pv == 0.0:
        out["v_eff"] = 0.0
    elif fs is None or _is_nan(fs):
        out["v_eff"] = None
    else:
        out["v_eff"] = pv * (1.0 - fs / 100.0)
    return out


def _fmt(value, kind):
    if value is None or _is_nan(value):
        return MISSING
    if kind == "pct":
        return f"{value:.2f}"
    if kind == "vol":
        return f"{value:.2f}"
    return str(value)


def _round_for_compare(value, kind):
    """Round to display precision so ties on the printed values bold together."""
    if value is None or _is_nan(value):
        return None
    if kind in ("pct", "vol"):
        return round(value, 2)
    return value


def _best_indices(values_by_method, kind, direction):
    """Return the set of method indices that achieve the best (rounded) value.

    Ties produce multiple indices; missing values are excluded; an empty set
    is returned if every method is missing.
    """
    rounded = {i: _round_for_compare(v, kind) for i, v in values_by_method.items()}
    valid = {i: v for i, v in rounded.items() if v is not None}
    if not valid:
        return set()
    best = min(valid.values()) if direction == "min" else max(valid.values())
    return {i for i, v in valid.items() if v == best}


def build_results_table():
    """Assemble the table as a pandas DataFrame.

    Columns are grouped metric-outer / method-inner. Best values per
    (system, metric) row are wrapped in ``*...*`` markers so the LaTeX
    renderer can promote them to ``\\textbf{...}``.
    """
    metric_cols = []
    for _, display_label, _, _, _ in METRICS:
        for method_label, _ in METHODS:
            metric_cols.append((display_label, method_label))
    columns = pd.MultiIndex.from_tuples(
        [("System", ""), ("(nx, nu)", "")] + metric_cols
    )

    rows = []
    for system_key, display_name in SYSTEMS:
        nx, nu = _system_dims(system_key)
        dims = f"({nx}, {nu})" if nx is not None else MISSING

        per_method = [_load_validation(system_key, group_dir)
                      for _, group_dir in METHODS]

        row = [display_name, dims]
        for metric_key, _, _, kind, direction in METRICS:
            values_by_method = {i: m[metric_key] for i, m in enumerate(per_method)}
            best_idx = _best_indices(values_by_method, kind, direction)
            for m_idx in range(len(METHODS)):
                s = _fmt(values_by_method[m_idx], kind)
                if m_idx in best_idx and s != MISSING:
                    s = f"*{s}*"
                row.append(s)
        rows.append(row)

    return pd.DataFrame(rows, columns=columns)


_BOLD_RE = re.compile(r"\*([^*]+)\*")


def _to_latex_cell(s):
    """Convert ``*X*`` bold markers from the DataFrame into ``\\textbf{X}``."""
    return _BOLD_RE.sub(r"\\textbf{\1}", s)


def to_latex_table(df, font_size="\\small", caption="CBF validation results.", label="tab:cbf_results"):
    """Render `df` as a page-width LaTeX table.

    Columns are grouped metric-outer / method-inner. Bold markers from
    `build_results_table` are converted to ``\\textbf{...}``.
    """
    n_methods = len(METHODS)
    n_metrics = len(METRICS)
    col_spec = (
        "@{\\extracolsep{\\fill}}lc"
        + (" " + "c" * n_methods) * n_metrics
        + "@{}"
    )

    header_top = "\\multirow{2}{*}{System} & \\multirow{2}{*}{($n_x$, $n_u$)}"
    for _, _, latex_label, _, _ in METRICS:
        header_top += f" & \\multicolumn{{{n_methods}}}{{c}}{{{latex_label}}}"
    header_top += " \\\\"

    cmidrules = ""
    for i in range(n_metrics):
        lo = 3 + i * n_methods
        hi = lo + n_methods - 1
        cmidrules += f" \\cmidrule(lr){{{lo}-{hi}}}"
    cmidrules = cmidrules.lstrip()

    method_names = [name for name, _ in METHODS]
    sub_headers = ["", ""] + method_names * n_metrics
    header_bot = " & ".join(sub_headers) + " \\\\"

    body_lines = []
    for _, row in df.iterrows():
        cells = [_to_latex_cell(str(v)) for v in row.tolist()]
        body_lines.append(" & ".join(cells) + " \\\\")
    body = "\n    ".join(body_lines)

    latex = (
        "\\begin{table*}[t]\n"
        "  \\centering\n"
        f"  \\caption{{{caption}}}\n"
        f"  \\label{{{label}}}\n"
        f"  {font_size}\n"
        "  \\setlength{\\tabcolsep}{2pt}\n"
        f"  \\begin{{tabular*}}{{\\textwidth}}{{{col_spec}}}\n"
        "    \\toprule\n"
        f"    {header_top}\n"
        f"    {cmidrules}\n"
        f"    {header_bot}\n"
        "    \\midrule\n"
        f"    {body}\n"
        "    \\bottomrule\n"
        "  \\end{tabular*}\n"
        "\\end{table*}\n"
    )
    return latex


In [ ]:
df = build_results_table()
with pd.option_context("display.max_columns", None, "display.width", 200):
    print(df.to_string(index=False))


In [ ]:
latex = to_latex_table(
    df,
    font_size="\\scriptsize",  # change to \footnotesize / \scriptsize as needed
    caption=(
        "CBF validation results --- effective safe volume "
        "$\\eta_\\mathrm{eff} = (\\operatorname{vol}(\\hat{\\mathcal{S}})/\\operatorname{vol}(\\mathcal{X}))(1 - \\rho_\\mathrm{FS})$, "
        "false-safe rate $\\rho_\\mathrm{FS}$ and false-unsafe rate "
        "$\\rho_\\mathrm{FU}$."
    ),
    label="tab:cbf_results",
)
print(latex)
